In [1]:
import json

In [2]:
from snowflake.snowpark import Session
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.snowpark.functions import col
from snowflake.ml.modeling import preprocessing

In [15]:
creds_path = r"C:\temp\sf_creds_temp.txt"
with open(creds_path,"r") as f:
    sf_creds = json.load(f)

In [5]:
# Here’s the neighborhood visiting pattern the truck follows:
# In January, the truck goes to N1 on the 1st, 8th, 15th, 22nd, and 29th, and N2 the other days.

# From February through November, it goes to:
# N1 on the 1st
# N2 on the 2nd
# N3 on the 3rd
# N4 on the 4th
# N5 on the 5th
# N6 on the 6th
# N7 on the 7th
# N1 on the 8th
# N2 on the 9th
# etc.

# Every December, it only goes to N8.

In [6]:
month_days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

pre = {}

In [7]:
for i,month_length in enumerate(month_days):
    month = i + 1

    for day in range(1,month_length+1):
        
        # In January, it goes to neighborhood 1 on Mondays, and neighborhood 2 the other days.
        if ((month) == 1):
            if (day) % 7 == 1:
                pre[(month,day)] = 1
            else:
                pre[(month,day)] = 2
                
        # From February through November, it goes to neighborhood 1 on the 1st, 2 on the 2nd, 3 on the 3rd,
        # 4 on the 4th, 5 on the 5th, 6 on the 6th, and 7 on the 7th, 1 on the 8th, 2 on the 9th, etc.
        elif ((month) <= 11):
            pre[(month,day)] = ((day-1) % 7) + 1

        # Every December, it only goes to neighborhood 8.
        elif ((month) == 12):
            pre[(month,day)] = 8


In [34]:
data = []
for x in range(20):
    for day, n in pre.items():
        data.append((day[0],day[1],n))

In [35]:
df_schema = ["MONTH","DAY","NEIGHBORHOOD"]

In [36]:
# create a Session with the necessary connection info
session = Session.builder.configs(sf_creds).create()

In [37]:
df_clean = session.create_dataframe(data, schema = df_schema)

In [38]:
df_clean.write.save_as_table("test_database.test_schema.df_clean")

In [39]:
# create a dataframe (though note that this doesn’t pull data into your local machine)
snowpark_df = session.table("test_database.test_schema.df_clean")


In [40]:
snowpark_df.show(n=40)

------------------------------------
|"MONTH"  |"DAY"  |"NEIGHBORHOOD"  |
------------------------------------
|1        |1      |1               |
|12       |31     |8               |
|12       |30     |8               |
|12       |29     |8               |
|12       |28     |8               |
|12       |27     |8               |
|12       |26     |8               |
|12       |25     |8               |
|12       |24     |8               |
|12       |23     |8               |
|12       |22     |8               |
|12       |21     |8               |
|12       |20     |8               |
|12       |19     |8               |
|12       |18     |8               |
|12       |17     |8               |
|12       |16     |8               |
|12       |15     |8               |
|12       |14     |8               |
|12       |13     |8               |
|12       |12     |8               |
|12       |11     |8               |
|12       |10     |8               |
|12       |9      |8               |
|

In [41]:
snowpark_df.describe().show()

---------------------------------------------------------------------------
|"SUMMARY"  |"MONTH"             |"DAY"              |"NEIGHBORHOOD"      |
---------------------------------------------------------------------------
|count      |7300.0              |7300.0             |7300.0              |
|mean       |6.526027            |15.720548          |4.019178            |
|stddev     |3.4480874408866145  |8.796849549696756  |2.2738080393911884  |
|min        |1.0                 |1.0                |1.0                 |
|max        |12.0                |31.0               |8.0                 |
---------------------------------------------------------------------------



In [43]:
# groupby neighborhood, and show the counts
snowpark_df.orderBy("NEIGHBORHOOD").group_by("Neighborhood").count().show()


----------------------------
|"NEIGHBORHOOD"  |"COUNT"  |
----------------------------
|1               |1080     |
|2               |1500     |
|4               |800      |
|3               |900      |
|5               |800      |
|6               |800      |
|7               |800      |
|8               |620      |
----------------------------



In [32]:
test = snowpark_df.withColumn('NEIGHBORHOOD2', snowpark_df.neighborhood - 1).drop("Neighborhood")


In [33]:
test.show()

-------------------------------------
|"MONTH"  |"DAY"  |"NEIGHBORHOOD2"  |
-------------------------------------
|1        |1      |0                |
|12       |31     |7                |
|12       |30     |7                |
|12       |29     |7                |
|12       |28     |7                |
|12       |27     |7                |
|12       |26     |7                |
|12       |25     |7                |
|12       |24     |7                |
|12       |23     |7                |
-------------------------------------

